# MPS State Preparation Resource Estimation Comparison

This notebook compares resource estimates for MPS (Matrix Product State) preparation using the following implementations:

1. **Dense** (`MPSSequentialStatePreparation`): Uses a CSD- and Givens-rotation-based decomposition.
2. **Sparse** (`MPSSparseStatePreparation`): Exploits the U(1) block sparsity of MPS tensors through a permutation-based decomposition.

Please download the MPS datasets (licensed under CC BY 4.0) published by Felix Rupprecht on [Zenodo](https://zenodo.org/records/20393500). 
The archives `p450_enzyme_G-1.tar.gz` and `fe2s2-2_small.tar.gz` should be placed in the `/data` directory.

**References**
1. Felix Rupprecht and Sabine Wölk. (2026). Faster matrix product state preparation by
exploiting symmetry-induced block-sparsity. arXiv:2605.28489. Zenodo: https://zenodo.org/records/20393500.

2. Dominic W. Berry et al. (2025). Rapid Initial-State Preparation for the Quantum Simulation of
Strongly Correlated Molecules. PRX Quantum 6, 020327.
https://doi.org/10.1103/PRXQuantum.6.020327.


In [ ]:
import sys
import glob
from pathlib import Path
import numpy as np
from scipy.sparse import load_npz

from qdk_chemistry.data.mps_wavefunction import MPSWavefunction
from qdk_chemistry.algorithms.state_preparation import MPSSequentialStatePreparation
from qdk_chemistry.algorithms.state_preparation.mps_sparse import MPSSparseStatePreparation

## Load MPS tensors into MPSWavefunction
The `fe2s2-2_small` dataset contains 20 MPS tensors (compressed to bond dimension
≤ 1000).

In [ ]:
COMPRESSED_PATH = Path("fe2s2-2_small") / "tensors_compressed" / "tensors_compressed_"
PHASE_BITS = 15 # number of qubits for phase gradient rotation

n_tensors = len(glob.glob(str(COMPRESSED_PATH) + "[0-9]*.npz"))
sparse_tensors = [load_npz(f"{COMPRESSED_PATH}{i}.npz") for i in range(n_tensors)]
dense_tensors = [t.toarray() for t in sparse_tensors]
mps_full = MPSWavefunction(dense_tensors)
print(f"MPSWavefunction: {mps_full.num_sites} sites, {mps_full.num_qubits} qubits")
print(f"Bond dims: {mps_full.bond_dims}")

## QDK-Chemistry MPS State Preparation

We generate the MPS state preparation circuit and run logical resource estimation.

This cell takes about 5 minutes to run.

In [ ]:
# Dense state preparation
algo_dense = MPSSequentialStatePreparation()
algo_dense.settings().set("rotation_bits", PHASE_BITS)
algo_dense.settings().set("fast_resource_estimation", True)

circuit_dense = algo_dense.run(mps_full)
result_dense = circuit_dense.estimate()
qdk_dense_counts = result_dense.logical_counts
print(f"\nQDK Dense logical counts: {qdk_dense_counts}")

# Sparse state preparation
algo_sparse = MPSSparseStatePreparation()
algo_sparse.settings().set("rotation_bits", PHASE_BITS)

circuit_sparse = algo_sparse.run(mps_full)
result_sparse = circuit_sparse.estimate()
qdk_sparse_counts = result_sparse.logical_counts
print(f"QDK Sparse logical counts: {qdk_sparse_counts}")

## QREv3

Here, we run physical resource estimation for the MPS state preparation circuit using QREv3. We use a Majorana-based architecture with an error rate of $10^{-5}$ and the measurement-based QEC code, along with round-based magic state factories. 

This cell takes about 3 minutes to run.

In [ ]:
from qdk.qre import estimate, LatticeSurgery, PSSPC
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

# Majorana-based architecture with an error rate of $10^{-5}$ and the `ThreeAux` 
# QEC code, along with round-based magic state factories
architecture = Majorana(error_rate=1e-5)
isa_query = ThreeAux.q() * RoundBasedFactory.q(use_cache=True, code_query=ThreeAux.q())
trace_query = (
    PSSPC.q()
    * LatticeSurgery.q(slow_down_factor=[1.0 * j for j in range(1, 20)])
)
# For QDK Chemistry circuit
app = circuit_dense.get_qre_application()
results = estimate(app, architecture, isa_query, trace_query, max_error=0.01, name="MPS Dense")
display(results.as_frame())